In [2]:
# Install core and orchestration libraries
!pip install --quiet "numpy<2.0.0" paddlepaddle==2.6.2 paddleocr==2.8.1 pymupdf opencv-python-headless numpy ollama pydantic pillow matplotlib
!pip install --quiet langchain-core langchain-ollama langgraph

In [3]:
!sudo apt-get install -y zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.


In [4]:
!curl -fsSL https://ollama.com/install.sh | sh
# Start Ollama server in the background
!nohup ollama serve &
!sleep 10 # Give Ollama server time to start

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
nohup: appending output to 'nohup.out'


In [5]:
!ollama pull qwen2.5vl:7b

In [7]:
import ollama

model_list = ollama.list()
print(model_list)

models=[Model(model='qwen2.5vl:7b', modified_at=datetime.datetime(2026, 6, 26, 8, 37, 0, 985657, tzinfo=TzInfo(0)), digest='5ced39dfa4bac325dc183dd1e4febaa1c46b3ea28bce48896c8e69c1e79611cc', size=5969245856, details=ModelDetails(parent_model='', format='gguf', family='qwen25vl', families=['qwen25vl'], parameter_size='8.3B', quantization_level='Q4_K_M'))]


In [8]:
import os
import sys
import cv2
import json
import base64
import logging
import numpy as np
import fitz  # PyMuPDF
from PIL import Image, ImageDraw
from typing import List, Optional, Dict, Any, Tuple, Literal
from pydantic import BaseModel, Field, ValidationError

# LangChain & LangGraph Orchestration Imports
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, END

# Configure centralized tracking logs
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger("EnterpriseSelfHealingPipeline")
logging.getLogger("ppocr").setLevel(logging.ERROR)

from paddleocr import PaddleOCR

# =====================================================================
# 1. Unified Enterprise Storage Schema
# =====================================================================
class UnifiedClaimSchema(BaseModel):
    """The unified data structure expected by our FastAPI enterprise database routing layer."""
    document_type: str = Field(..., description="Classification mapping: 'Hospital Bill', 'Prescription', 'Lab Report', or 'Discharge Summary'")
    institution_name: str = Field(..., description="The name of the issuing hospital, clinic, pharmacy, or laboratory facility.")
    patient_name: Optional[str] = None
    service_date: Optional[str] = None
    clinical_summary: str = Field(..., description="Aggregated text tracking primary diagnoses, findings, reasons for visit, or symptoms.")
    line_items_or_treatments: List[str] = Field(..., description="Array of itemized procedures, medications, or lab test metrics found.")
    monetary_total: Optional[float] = Field(None, description="Total balance due or payment total. Null if not applicable to document type.")

# =====================================================================
# 2. LangGraph Orchestration State Definitions
# =====================================================================
class PipelineState(BaseModel):
    """Defines the operational memory state flowing through the self-healing loop graph."""
    clean_matrix: Any  # np.ndarray image matrix
    ocr_text: str
    b64_image: str
    retry_count: int = 0
    max_retries: int = 2
    raw_vlm_output: str = ""
    error_message: str = ""
    validated_record: Optional[Dict[str, Any]] = None
    pipeline_success: bool = False

# =====================================================================
# 3. Core Enterprise Pipeline Engine Module
# =====================================================================
class SelfHealingClaimsPipeline:
    def __init__(self, use_gpu: bool = False):
        logger.info("Initializing industrial PaddleOCR engine instance...")
        self.ocr_engine = PaddleOCR(use_angle_cls=True, lang='en', use_gpu=use_gpu)

        # Initialize native LangChain chat model wrapper bound to local Ollama runtime
        self.llm = ChatOllama(model="qwen2.5vl:7b", temperature=0.0)
        self.graph = self._build_self_healing_graph()

    def _convert_pdf_to_images(self, pdf_path: str, dpi: int = 150) -> List[np.ndarray]:
        doc = fitz.open(pdf_path)
        images = []
        zoom = dpi / 72
        matrix = fitz.Matrix(zoom, zoom)
        for page in doc:
            pix = page.get_pixmap(matrix=matrix, alpha=False)
            img_data = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.h, pix.w, pix.n)
            images.append(cv2.cvtColor(img_data, cv2.COLOR_RGB2BGR))
        doc.close()
        return images

    def _preprocess_image(self, img: np.ndarray) -> np.ndarray:
        if len(img.shape) == 3:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        else:
            gray = img
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        contrast_enhanced = clahe.apply(gray)
        denoised = cv2.medianBlur(contrast_enhanced, 3)
        _, binary = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        coords = np.column_stack(np.where(cv2.bitwise_not(binary) > 0))
        if coords.size > 0:
            angle = cv2.minAreaRect(coords)[-1]
            angle = -(90 + angle) if angle < -45 else -angle
            h, w = img.shape[:2]
            rot_matrix = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
            img = cv2.warpAffine(img, rot_matrix, (w, h), flags=cv2.INTER_CUBIC, borderValue=(255, 255, 255))
        return img

    def _run_ocr(self, img: np.ndarray) -> str:
        raw_results = self.ocr_engine.ocr(img, cls=True)
        text_lines = []
        if raw_results and raw_results[0]:
            for block in raw_results[0]:
                text_lines.append(block[1][0])
        return "\n".join(text_lines)

    # -----------------------------------------------------------------
    # LangGraph State Node Functions
    # -----------------------------------------------------------------
    def node_extraction_agent(self, state: PipelineState) -> Dict[str, Any]:
        """Node 1: Executes baseline structured schema visual query logic extraction."""
        logger.info(f"🤖 [Node: Extraction] Querying Qwen2.5-VL (Attempt {state.retry_count + 1})...")

        system_prompt = (
            "You are an enterprise medical insurance ingestion agent.\n"
            "Analyze the document layout image and the provided OCR text block to extract structured metrics.\n"
            "You must return ONLY a raw JSON payload fitting this exact structure:\n"
            "{\n"
            '  "document_type": "Hospital Bill | Prescription | Lab Report | Discharge Summary",\n'
            '  "institution_name": "string",\n'
            '  "patient_name": "string",\n'
            '  "service_date": "YYYY-MM-DD",\n'
            '  "clinical_summary": "string describing diagnosis or symptoms summary",\n'
            '  "line_items_or_treatments": ["string items"],\n'
            '  "monetary_total": 0.0 or null\n'
            "}\n"
            "CRITICAL: Do not include markdown wraps like ```json. Do not include conversational commentary."
        )

        # Format the data URI standard structure for ChatOllama multimodal interface
        image_content_uri = f"data:image/png;base64,{state.b64_image}"

        messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=[
                {"type": "text", "text": f"Extract details based on this verified OCR reference block:\n{state.ocr_text}"},
                {"type": "image_url", "image_url": {"url": image_content_uri}}
            ])
        ]

        response = self.llm.invoke(messages)
        return {"raw_vlm_output": response.content.strip()}

    def node_validation_guard(self, state: PipelineState) -> Dict[str, Any]:
        """Node 2: Structural validation logic block filtering incoming text outputs."""
        logger.info("🔍 [Node: Validation] Validating structure and payload constraints...")
        raw_text = state.raw_vlm_output

        # Clean potential markdown block formatting leak issues
        if raw_text.startswith("```json"):
            raw_text = raw_text[7:]
        if raw_text.endswith("```"):
            raw_text = raw_text[:-3]
        raw_text = raw_text.strip()

        try:
            parsed_json = json.loads(raw_text)
            # Enforce validation matching defined pydantic model schema rules
            validated_model = UnifiedClaimSchema(**parsed_json)
            logger.info("✅ Structural compliance checks passed perfectly.")
            return {
                "validated_record": validated_model.model_dump(),
                "pipeline_success": True,
                "error_message": ""
            }
        except (json.JSONDecodeError, ValidationError) as err:
            logger.warning(f"⚠️ Structural integrity checks failed: {str(err)}")
            return {
                "error_message": str(err),
                "pipeline_success": False
            }

    def node_remediation_agent(self, state: PipelineState) -> Dict[str, Any]:
        """Node 3: Formulates focused corrective critique prompts if exceptions occur."""
        logger.info(f"🔄 [Node: Remediation] Healing broken schema state (Errors logged: {state.retry_count + 1})")

        remediation_prompt = (
            f"Your previous JSON output failed validation rules and crashed the processing engine.\n"
            f"CRITICAL ENGINE FAULT BREAKDOWN:\n{state.error_message}\n\n"
            f"FAULTY PAYLOAD RECEIVED:\n{state.raw_vlm_output}\n\n"
            f"Fix this object schema structure. Regenerate and fix the fields. Return ONLY raw valid JSON matching standard target types."
        )

        image_content_uri = f"data:image/png;base64,{state.b64_image}"
        messages = [
            SystemMessage(content="You are a validation error remediation sub-agent. Fix JSON schemas so they match requirements accurately."),
            HumanMessage(content=[
                {"type": "text", "text": remediation_prompt},
                {"type": "image_url", "image_url": {"url": image_content_uri}}
            ])
        ]

        response = self.llm.invoke(messages)
        return {
            "raw_vlm_output": response.content.strip(),
            "retry_count": state.retry_count + 1
        }

    # -----------------------------------------------------------------
    # LangGraph Compilation Strategy
    # -----------------------------------------------------------------
    def _build_self_healing_graph(self):
        builder = StateGraph(PipelineState)

        # Add processing units as executable network nodes
        builder.add_node("extraction_agent", self.node_extraction_agent)
        builder.add_node("validation_guard", self.node_validation_guard)
        builder.add_node("remediation_agent", self.node_remediation_agent)

        # Set orchestrator trace pathway configuration
        builder.set_entry_point("extraction_agent")
        builder.add_edge("extraction_agent", "validation_guard")

        # Define condition logic route pathways
        def route_condition(state: PipelineState) -> Literal["success", "retry", "abort"]:
            if state.pipeline_success:
                return "success"
            if state.retry_count < state.max_retries:
                return "retry"
            return "abort"

        builder.add_conditional_edges(
            "validation_guard",
            route_condition,
            {
                "success": END,
                "retry": "remediation_agent",
                "abort": END
            }
        )
        builder.add_edge("remediation_agent", "validation_guard")
        return builder.compile()

    # -----------------------------------------------------------------
    # End-to-End Orchestrator Pipeline Interface Loop
    # -----------------------------------------------------------------
    def execute_pipeline(self, file_path: str) -> List[Dict[str, Any]]:
        logger.info(f"🚀 Initializing processing execution lifecycle for: {file_path}")
        ext = os.path.splitext(file_path)[1].lower()
        pages = self._convert_pdf_to_images(file_path) if ext == '.pdf' else [cv2.imread(file_path)]

        validated_batch_records = []

        for idx, raw_matrix in enumerate(pages):
            if raw_matrix is None: continue
            logger.info(f"Processing Page Context Index {idx+1}/{len(pages)}")

            clean_matrix = self._preprocess_image(raw_matrix)
            ocr_text = self._run_ocr(clean_matrix)

            _, buffer = cv2.imencode('.png', clean_matrix)
            b64_image = base64.b64encode(buffer).decode('utf-8')

            # Setup State Machine Context memory tracking objects
            initial_state = PipelineState(
                clean_matrix=clean_matrix,
                ocr_text=ocr_text,
                b64_image=b64_image,
                retry_count=0
            )

            # Run State Machine until final graph completion
            final_output_state = self.graph.invoke(initial_state)

            if final_output_state.get("pipeline_success"):
                validated_batch_records.append(final_output_state["validated_record"])
            else:
                logger.error(f"❌ Document page analysis pipeline failed structural ingestion validation. Skipping.")

        return validated_batch_records

# =====================================================================
# 4. Simulation Execution Tests Harness
# =====================================================================
def generate_synthetic_corpus():
    """Generates a sample corpus directly to disk for processing validation checks."""
    rx_img = Image.new('RGB', (650, 250), color=(255, 255, 255))
    draw = ImageDraw.Draw(rx_img)
    draw.text((30, 20), "OAKRIDGE PHARMACY & HEALTHCARE", fill=(0,0,0))
    draw.text((30, 60), "Patient: John Harrison", fill=(0,0,0))
    draw.text((30, 90), "Rx: Atorvastatin 20mg Tablets (Take 1 daily at bedtime)", fill=(0,0,0))
    draw.text((30, 120), "Dr. Amanda Ross, MD - Date: 2026-01-15", fill=(0,0,0))
    rx_img.save("prescription_claim.png")

    pdf = fitz.open()
    p = pdf.new_page(width=600, height=300)
    p.insert_text((40, 40), "ST. JUDE CLINICAL CENTER INVOICE", fontsize=16)
    p.insert_text((40, 90), "Patient Target Account: John Harrison")
    p.insert_text((40, 120), "Inpatient Facility Day Stay Charges:  $1500.00")
    p.insert_text((40, 160), "AGGREGATED BALANCE OWED PAYABLE:      $1500.00")
    p.insert_text((40, 200), "Billing Closeout Date Field: 2026-03-01")
    pdf.save("hospital_bill.pdf")
    pdf.close()
    print("📁 Claims mock files written to local storage volume loops.")

if __name__ == "__main__":
    generate_synthetic_corpus()

    # Instantiate the processing pipeline engine
    pipeline_engine = SelfHealingClaimsPipeline(use_gpu=False)

    target_queue = ["prescription_claim.png", "hospital_bill.pdf"]
    master_records_database = []

    print("\n--- Starting Self-Healing Operational Loops ---")
    for file_target in target_queue:
        try:
            records = pipeline_engine.execute_pipeline(file_target)
            master_records_database.extend(records)
        except Exception as failure:
            print(f"CRITICAL FAULT: Processing aborted for target context: {str(failure)}")

    print("\n========================= INGESTION RUN FINISHED =========================")
    print(json.dumps(master_records_database, indent=2))

📁 Claims mock files written to local storage volume loops.
download https://paddleocr.bj.bcebos.com/PP-OCRv3/english/en_PP-OCRv3_det_infer.tar to /root/.paddleocr/whl/det/en/en_PP-OCRv3_det_infer/en_PP-OCRv3_det_infer.tar


100%|██████████| 4.00M/4.00M [00:00<00:00, 6.05MiB/s]


download https://paddleocr.bj.bcebos.com/PP-OCRv4/english/en_PP-OCRv4_rec_infer.tar to /root/.paddleocr/whl/rec/en/en_PP-OCRv4_rec_infer/en_PP-OCRv4_rec_infer.tar


100%|██████████| 10.2M/10.2M [00:00<00:00, 10.6MiB/s]


download https://paddleocr.bj.bcebos.com/dygraph_v2.0/ch/ch_ppocr_mobile_v2.0_cls_infer.tar to /root/.paddleocr/whl/cls/ch_ppocr_mobile_v2.0_cls_infer/ch_ppocr_mobile_v2.0_cls_infer.tar


100%|██████████| 2.19M/2.19M [00:00<00:00, 3.39MiB/s]


[2026/06/26 08:38:14] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, use_mlu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, gpu_id=0, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='/root/.paddleocr/whl/det/en/en_PP-OCRv3_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='/root/.paddleocr/whl/rec/en/en_PP-OCRv4_rec_infer', rec_image_inverse=True, rec_image_shape='3, 48, 320', rec_batch_num=6, max_text_length=25, rec_c